In [1]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

from pm.data import download_prices
from pm.returns import simple_returns

In [2]:
'''
These tickers were part of my portfolio composition until August 2025, since that some positions have been closed or some other new ones have been opened due to earnins reports,
economic trends or just interest in different companies with higher risk/return expectations. As mentioned, this list of tickers is just an example of an stale potfolio
which can be improved 
'''

tickers = ['NVDA','PLTR','VOO','QQQ','IBIT','GLDM','AVGO','META','MSFT','GOOGL','UBER','WMT','V','MA','SPOT','SOFI','QCOM','BABA','TSM']

end_date = datetime.today()
start_date = end_date - timedelta(days = 5*365)
price_data = 'Close'

port = download_prices(tickers,price_data,start_date,end_date)
port

,NVDA,PLTR,VOO,QQQ,IBIT,GLDM,AVGO,META,MSFT,GOOGL,UBER,WMT,V,MA,SPOT,SOFI,QCOM,BABA,TSM
Date,,,,,,,,,,,,,,,,,,,
2020-12-28,12.863822,25.629999,318.437744,303.433411,NaN,37.299999,38.852245,275.078522,215.858047,88.032852,51.970001,45.226540,205.057816,334.853180,317.290009,NaN,132.655182,211.174179,97.610863
2020-12-29,12.906952,24.660000,317.767700,303.705170,NaN,37.439999,38.596760,274.860016,215.080826,87.228928,52.270000,44.940010,206.735825,336.981964,318.429993,NaN,133.328644,224.374924,97.068336
2020-12-30,13.108885,25.100000,318.195770,303.714844,NaN,37.720001,39.117638,269.984100,212.710739,86.161484,53.150002,44.902645,210.583740,345.623749,319.350006,NaN,134.639542,226.397797,100.139656
2020-12-31,13.018390,23.549999,319.908478,304.462158,NaN,37.880001,39.389317,271.265167,213.420837,86.974846,51.000000,44.893299,210.940567,346.974945,314.660004,NaN,136.785522,221.022507,100.268387
2021-01-04,13.076727,23.370001,315.570892,300.163177,NaN,38.720001,38.253109,267.074432,208.882248,85.659279,51.139999,45.634514,210.005096,341.677063,311.000000,12.200000,133.337601,216.388016,102.714401
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-17,170.940002,177.289993,615.607788,599.637390,48.709999,85.919998,325.397369,649.500000,476.119995,296.720001,79.139999,115.660004,344.410004,565.469971,566.229980,25.270000,172.339996,147.089996,276.959991
2025-12-18,174.139999,185.690002,620.254700,608.326233,47.959999,85.779999,329.250031,664.450012,483.980011,302.459991,79.690002,114.830002,346.010010,566.210022,563.820007,26.290001,174.190002,147.320007,284.679993
2025-12-19,180.990005,193.380005,625.789001,616.255981,49.910000,85.900002,339.709991,658.770020,485.920013,307.160004,79.309998,114.360001,349.250000,572.229980,582.159973,27.240000,175.250000,149.789993,288.950012


## Simple vs Logarithmic Returns

### Definitions

Let $P_t$ denote the price (or value) of an asset at the end of period $t$.

#### Simple (Arithmetic) Return
$$
R_t = \frac{P_t}{P_{t-1}} - 1
$$

- Interpreted as the **percentage gain or loss** over the period.
- For example, $R_t = 0.02$ corresponds to a **2% return**.

#### Logarithmic (Continuously Compounded) Return
$$
r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)
$$

- Defined as the logarithm of the gross return.
- For small returns, simple and log returns are numerically very close.

The exact relationship between both measures is:
$$
r_t = \ln(1 + R_t), \qquad R_t = e^{r_t} - 1
$$

---

### Time Aggregation

A key difference between both return definitions lies in their aggregation over time.

Log-returns are **additive** across periods:
$$
r_{1:T} = \sum_{t=1}^{T} r_t
$$

This property makes log-returns convenient for multi-period analysis and certain theoretical models.

In contrast, simple returns compound **multiplicatively**:
$$
1 + R_{1:T} = \prod_{t=1}^{T} (1 + R_t)
$$

---

### Portfolio Returns and Performance Measurement

For a portfolio rebalanced at the beginning of each period, the portfolio value evolves as:
$$
V_t = V_{t-1}(1 + R_t^{(p)})
$$

where the portfolio return is given by a weighted sum of individual asset returns:
$$
R_t^{(p)} = \sum_i w_i R_{i,t}
$$

This linearity property holds **only for simple returns** and is fundamental in portfolio construction, performance attribution, and risk measurement.

Log-returns do not satisfy this property in general:
$$
\ln(1 + R_t^{(p)}) \neq \sum_i w_i \ln(1 + R_{i,t})
$$

---

### Practical Takeaway

- Portfolio wealth, P&L, cumulative returns, and drawdowns are naturally defined using **simple returns**.
- Log-returns are primarily useful for **time aggregation** and **continuous-time modeling**.
- For time-series analysis (like GARCH OR GBM), I'd choose **Log Returns** but since this project is about **allocating capital across assets**, using exclusively **Simple Returns** is the only way to ensure the portfolio math actually adds up as expected. 

> **"If you are tracking money, use simple returns."**

## Methodological Choice

In quantitative finance, there is a well-known trade-off between using **Log Returns** (ideal for time-aggregation and modelling) and **Simple Returns** (better for cross-sectional aggregation).
Given that this project relies on **Modern Portfolio Theory (MPT)**, the use of simple returns is a deliberate methodological choice. MPT relies on the linear aggregation of asset returns through portfolio weights, where the portfolio return is defined as a weighted sum of individual asset returns. This linearity holds for simple returns but not for log returns, which are not additive across assets. While log-returns are mathematically elegant for time-series modeling, they break the linear relationship required for portfolio weights. For this reason, and to remain consistent with the assumptions of MPT when computing portfolio returns, risk, and correlations, this project uses **Simple Returns** throughout.

In [3]:
simple_ret = simple_returns(port)
simple_ret

,NVDA,PLTR,VOO,QQQ,IBIT,GLDM,AVGO,META,MSFT,GOOGL,UBER,WMT,V,MA,SPOT,SOFI,QCOM,BABA,TSM
Date,,,,,,,,,,,,,,,,,,,
2020-12-29,0.003353,-0.037846,-0.002104,0.000896,NaN,0.003753,-0.006576,-0.000794,-0.003601,-0.009132,0.005773,-0.006335,0.008183,0.006357,0.003593,NaN,0.005077,0.062511,-0.005558
2020-12-30,0.015645,0.017843,0.001347,0.000032,NaN,0.007479,0.013495,-0.017740,-0.011020,-0.012237,0.016836,-0.000831,0.018613,0.025645,0.002889,NaN,0.009832,0.009016,0.031641
2020-12-31,-0.006903,-0.061753,0.005383,0.002461,NaN,0.004242,0.006945,0.004745,0.003338,0.009440,-0.040452,-0.000208,0.001694,0.003909,-0.014686,NaN,0.015939,-0.023743,0.001286
2021-01-04,0.004481,-0.007643,-0.013559,-0.014120,NaN,0.022175,-0.028846,-0.015449,-0.021266,-0.015126,0.002745,0.016511,-0.004435,-0.015269,-0.011632,NaN,-0.025207,-0.020968,0.024395
2021-01-05,0.022210,0.052632,0.006578,0.008244,NaN,0.002583,0.006773,0.007548,0.000964,0.008064,0.056120,-0.005323,-0.014925,-0.011579,0.008746,-0.004098,0.026465,0.055080,0.009579
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-17,-0.038150,-0.055712,-0.010990,-0.018537,-0.020117,0.008214,-0.044770,-0.011641,-0.000567,-0.032130,-0.021997,0.002079,-0.002028,-0.000972,-0.022174,-0.049285,-0.021463,-0.014736,-0.034545
2025-12-18,0.018720,0.047380,0.007548,0.014490,-0.015397,-0.001629,0.011840,0.023018,0.016508,0.019345,0.006950,-0.007176,0.004646,0.001309,-0.004256,0.040364,0.010735,0.001564,0.027874
2025-12-19,0.039336,0.041413,0.008923,0.013035,0.040659,0.001399,0.031769,-0.008548,0.004008,0.015539,-0.004769,-0.004093,0.009364,0.010632,0.032528,0.036135,0.006085,0.016766,0.014999
